1. Importing Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
import tensorflow as tf

2. Load Data

In [2]:
# dataset path
DATA_PATH = "../../annotations"
os.listdir(DATA_PATH)

['lisa_dataset_split.csv']

In [3]:
# Load the precomputed split.
# Expects 'lisa_dataset_split.csv' to be located under DATA_PATH.
df = pd.read_csv(os.path.join(DATA_PATH,'lisa_dataset_split.csv'), sep=';')

In [4]:
# Preview first 5 rows
df.head()

,image_id,label,x_min,y_min,x_max,y_max,frame,isNight,clipNames,split
0,../../datasets/lisa-traffic-light-dataset\dayT...,1,698,333,710,358,0,0,dayClip1,train
1,../../datasets/lisa-traffic-light-dataset\dayT...,1,846,391,858,411,0,0,dayClip1,train
2,../../datasets/lisa-traffic-light-dataset\dayT...,1,698,337,710,357,1,0,dayClip1,train
3,../../datasets/lisa-traffic-light-dataset\dayT...,1,847,390,859,410,1,0,dayClip1,train
4,../../datasets/lisa-traffic-light-dataset\dayT...,1,698,331,710,356,2,0,dayClip1,train


In [5]:
# Shape: (rows, cols)
df.shape

(51826, 10)

In [6]:
#IMG_SIZE = 50  
IMG_SIZE = 50
output_folder = "../../datasets/processed_images"

# Buffers for data (images, labels, night-flag) and for the new CSV rows
X = []
y = []
z = []
new_rows = []

for idx, row in df.iterrows():
    img_path = row['image_id'].replace('\\', '/')

    # Skip if image file is missing
    if not os.path.exists(img_path):
        print(f"❌ Not found: {img_path}")
        continue

    try:
        img = cv2.imread(img_path)
        if img is None:
            # Skip unreadable images
            print(f"⚠️ Image read error: {img_path}")
            continue

        # Crop by bounding box
        x1, y1, x2, y2 = int(row['x_min']), int(row['y_min']), int(row['x_max']), int(row['y_max'])
        cropped = img[y1:y2, x1:x2]

        # Skip empty crops
        if cropped.size == 0:
            print(f"⚠️ Empty crop for image: {img_path}")
            continue

        # Resize and normalize to [0, 1]
        resized = cv2.resize(cropped, (IMG_SIZE, IMG_SIZE))
        norm_img = resized / 255.0

        # Create split subfolder (train/val/test)
        split_folder = os.path.join(output_folder, row['split'])
        os.makedirs(split_folder, exist_ok=True)

        # Save processed image
        new_img_name = f"img_{idx}.jpg"
        new_img_path = os.path.join(split_folder, new_img_name)
        cv2.imwrite(new_img_path, (norm_img * 255).astype(np.uint8))

        # Collect metadata for arrays and CSV
        X.append(norm_img)
        y.append(row['label'])
        z.append(row['isNight'])
        new_rows.append({
            'image_id': new_img_path,
            'label': row['label'],
            'isNight': row['isNight'],
            'split': row['split']
        })

    except Exception as e:
        # Log unexpected errors and continue
        print(f"⚠️ Exception for image {img_path}: {e}")
        continue

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)
z = np.array(z)

# Save new CSV with processed image paths and labels
if new_rows:
    new_df = pd.DataFrame(new_rows)
    new_df.to_csv("../../annotations/new_annotations.csv", sep=';', index=False)
    print(f"✅ Saved {len(new_rows)} images under '{output_folder}' and CSV to 'new_annotations.csv'")
else:
    print("⚠️ ΔNo images were saved.")

✅ Saved 51826 images under '../../datasets/processed_images' and CSV to 'new_annotations.csv'
